# ACCESS-AIS3 -- Stage 2 (scaffold): HO friction re-inversion

**Post-inversion extensions context:**

Standard practice after a friction inversion: higher-order thermal spin-up (SSA has
no vertical shear physics, so temperature/rheology can't be solved self-consistently
-- ais_0.1_param.py's surface-temperature-as-proxy approach is a known, flagged
caveat), a friction re-inversion under that higher-order physics (SSA-tuned friction
isn't valid once vertical shear resistance is added), ocean melt-rate calibration, a
short post-inversion relaxation, and a historical run tuned against observed dH/dt.

Stage 1 (`ho_thermal_steadystate`) is implemented and ready to test. Stages 2-5 are
scaffolds: correct model loading / solver setup / save-submit structure, each with an
explicit TODO marking the science decision that still needs iteration. Do not treat
their output as validated the way `AIS3_inverted.nc` / `AIS3_relaxed.nc` are once
stage 1 has actually been run and checked.

See `docs/inversion_worklog.md` and the originating plan for the full investigation.


## Imports & helper functions

In [ ]:
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import os


def friction_law_info(md):
    """Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law.

    Schoof (regularized Coulomb) inverts 'FrictionC' (field md.friction.C); Budd/Weertman (the
    'default' class) inverts 'FrictionCoefficient' (field md.friction.coefficient). The saved
    friction class (set by friction_law in ais_0.1_param.py) is the single source of truth.
    """
    if type(md.friction).__name__ == 'default':   # Budd / Weertman power law
        # VALIDATED bounds [0.05, 900] for p=q=1 (grounded RMSE 61.4 unregularised / 60.4 with
        # cf501=0.0001, see ais_0.1_param.py). The earlier [0.1, 10] bounds were tuned for the
        # superseded p=q=3 law (u ~ C^-6, `friconly_nfix`, RMSE 98.9) -- under p=q=1 (u ~ C^-2)
        # fast ice needs a much larger C for the same resisting stress, and that ceiling pinned
        # 41% of the domain at C=10 the first time p=1 was tried with it.
        return 'FrictionCoefficient', 'coefficient', 0.05, 900
    # Schoof (regularized Coulomb): tested directly for the Siple Coast trunk deficit and ruled
    # out on physics grounds, not just numerics -- see docs/inversion_worklog.md section 5.4.
    # Coupled-domain adjoint inversion of FrictionC is unstable near the Coulomb cap regardless
    # of solver settings, and a forward-only sweep across the full documented Cmax range
    # (0.17-0.84) left the Siple Coast trunk ratio completely unchanged from Budd's. Kept here
    # only so friction_law='schoof' remains loadable; not the recommended path.
    return 'FrictionC', 'C', 0.05, 250 ** 2        # Schoof (regularized Coulomb)


def extract_friction_inversion_domain(md):
    """Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition.

    Extracts ALL ice (ice_levelset_elements < 1, includes the ice-front elements), so the new
    mesh boundary coincides exactly with the true, contiguous ice margin -- not an arbitrary
    internal cut. extract() imposes Dirichlet (observed velocity) on every new boundary node by
    default (see Model.py: "Boundary conditions: Dirichlets on new boundary"); this is reverted
    to Neumann (NaN spc) at boundary nodes classified as floating (ocean_levelset < 0), i.e. true
    ice-shelf calving fronts, where the natural ocean-pressure BC is physically correct. Boundary
    nodes classified as grounded (ocean_levelset >= 0) -- both marine-terminating (bed below sea
    level, no shelf) and true land-terminating (bed above sea level, no ocean to push back
    against) -- keep extract()'s default Dirichlet, since Neumann has no obvious physical meaning
    there. Classification is per-vertex (mds.mask.ocean_levelset), not per-element, so it follows
    the true ice-front geometry exactly with no fragmentation.

    A prior version anchored only the Ronne-Filchner/Ross fronts (the two largest floating
    regions, found to blow up under pure Neumann at low friction coefficient) and left everything
    else -- including land-terminating margins -- as Neumann. That fixed Ronne-Filchner/Ross but
    left land-terminating margins with a physically meaningless Neumann BC, which was the actual
    cause of a ~1e10 m/yr blowup at coeff=1 (confirmed: switching those margins to Dirichlet here
    brought coeff=1 down to ~1.8e7 m/yr).
    """
    ice_levelset_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract(ice_levelset_elements < 1)

    bnd = mds.mesh.vertexonboundary.astype(bool)
    ocean_ls = np.asarray(mds.mask.ocean_levelset).ravel()
    floating_bnd = bnd & (ocean_ls < 0)

    mds.stressbalance.spcvx[floating_bnd] = np.nan
    mds.stressbalance.spcvy[floating_bnd] = np.nan
    mds.stressbalance.spcvz[floating_bnd] = np.nan
    mds.mask.ice_levelset[floating_bnd] = 0

    return mds


def load_shelf_rheology_B():
    """Load the validated floating-shelf rheology inversion result (extractedvertices, B).

    execution_newB_rheology/run_001_1_10_1e-17 (2026-07-20) supersedes
    models/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17 (2026-06-30, the
    `rheology_lcurve_run` config value): same regularisation point (cf101=1, cf103=10,
    cf502=1e-17), recomputed later in this project after several geometry/N-flooring fixes
    were developed (100m thickness floor, N re-flooring against it, etc.) -- run_004 predates
    those fixes. This is the actual source the validated grounded-RMSE-60.4 friction result
    was warm-started from; loading the stale run_004 instead was found (via a direct A/B
    test) to reproduce RMSE ~114, not ~60 -- see docs/inversion_worklog.md. Not yet promoted
    into the canonical models/AIS3_ssa_rheology_floating_inv_lcurve/ directory, so this loads
    it from its original ad-hoc execution directory via solve(load_only=True) instead of
    io.load_model().
    """
    _cl = pyissm.model.classes.cluster.gadi()
    _cl.codepath = os.environ['ISSM_DIR'] + '/bin'
    _cl.executionpath = '/g/data/au88/jh7060/ACCESS-AIS3/execution_newB_rheology'
    _cl.login = 'jh7060'; _cl.project = 'au88'; _cl.storage = 'gdata/au88'

    mshelf = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    mshelf.mask.ice_levelset = pyissm.model.param.kill_icebergs(mshelf)
    sel = (mshelf.mask.ocean_levelset < 0) & (mshelf.mask.ice_levelset < 0)
    mshelf = mshelf.extract(sel)
    mshelf.cluster = _cl
    mshelf.settings.waitonlock = 0
    mshelf.inversion.iscontrol = 0
    mshelf.miscellaneous.name = 'run_001_1_10_1e-17'
    mr = pyissm.model.execute.solve(mshelf, 'Stressbalance', load_only = True, runtime_name = False, check_consistency = False)
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## Configure options

In [ ]:
## ------------------------------------
## Configure options
## ------------------------------------

# Change directory to gdata to prevent storage limits in $HOME
os.chdir('/g/data/au88/jh7060/ACCESS-AIS3/')
os.environ['ISSM_DIR'] = '/g/data/vk83/apps/spack/1.1/release/linux-x86_64/issm-git.2026.05.18_2026.05.18-kgta35igm37z4qnqnul7rcmgx2inftqd'

# Should plots be generated?
plot = True
diagnostics = True
save = True
inversion_sensitivity = False

# Define execution directory
execution_dir = '/g/data/au88/jh7060/ACCESS-AIS3/execution'

# Define location to save final models
model_dir = '/g/data/au88/jh7060/ACCESS-AIS3/models'

# Define domain_file
domain_file = ('/g/data/au88/jh7060/ACCESS-AIS3/assets/ais_domain.exp')

# Define param_file
param_file = ('/g/data/au88/jh7060/ACCESS-AIS3/config/ais_0.1_param.py')

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']+'/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']  # was access-issm/2025.11.0: executing a
# 2026.05.18 binary under a 2025.11.0 module load -- a stale-module mismatch caught and fixed
# across every scratchpad script this session; production had not been updated to match.
# np/memory: 32 cores / 100GB is under-provisioned -- this mesh needs ~130GB minimum even at
# 32 ranks (see docs/inversion_worklog.md section 8), which is the likely real cause of the
# OOM history noted below on the maxsteps line, not maxsteps itself. 48 cores / 190GB is the
# configuration validated as SU-optimal this session (>96 cores was actively worse).
cluster.np = 48
cluster.memory = 190
cluster.time = 60*48
cluster.login = 'jh7060'
cluster.project = 'au88'


all_steps = [
    'process_domain',
    'mesh',
    'param',
    'ssa_rheology_floating_inv_sensit',
    'ssa_rheology_floating_inv_lcurve',
    # 'ssa_rheology_floating_inv',
    'ssa_friction_forward_check',
    'ssa_friction_forward_check_budd',
    'ssa_friction_inv_sensit',
    'ssa_friction_inv_lcurve',
    'ssa_friction_inv_reg_lcurve',
    'ssa_inverted_solve',
    'ssa_relaxation',
    'ho_thermal_steadystate',
    'ho_friction_inv',
    'melt_gamma_tuning',
    'ho_relaxation',
    'historical_dhdt_tuning',
]

# Define steps to run (this notebook is scoped to this group)
steps = ['ho_friction_inv']

## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)

In [ ]:
## ------------------------------------
## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)
## ------------------------------------
# Floating-ice rheology B field taken from the rheology L-curve (cf502 regularisation).
rheology_lcurve_run = 'run_004_1_10_1e-17'

# Preferred 101/103 cost-function coefficients for the friction inversion.
# The cf101=1000/cf103=0.1 choice below (run_021, vel_rmse=960.5) came from a sensit sweep run
# against the C_init=10 dead-zone bug (see ais_0.1_param.py): with u ~ C^-6 and the model stuck
# at zero velocity everywhere, that sweep's vel_rmse was never measuring model skill (it was
# ~equal to RMS(v_obs) itself, i.e. the null model). Every cell in that grid is void.
# VALIDATED instead (grounded RMSE 98.9, `friconly_nfix`): cf101=10, cf103=100 -- log-weighted,
# so the slow interior (which absolute weighting like 1000/0.1 effectively ignores) contributes
# to the fit. 10/100 was carried through every successful run this pipeline is based on.
friction_cf101 = 10
friction_cf103 = 100

# Mirrors friction_law in ais_0.1_param.py -- that flag only lives inside ais_0.1_param.py's
# own exec-scope (set on md.friction when 'param' in steps calls parameterize(), see below),
# so it isn't otherwise visible here at module load time where friction_lcurve_run (needed by
# ssa_inverted_solve) is defined. Keep this in sync with ais_0.1_param.py by hand.
friction_law = 'schoof'  # 'schoof' or 'budd' -- must match ais_0.1_param.py

# Effective-pressure source for the friction law. coupling=2 (ISSM internal "uniform sheet"
# hydrology, clamped >= 0) matched or beat coupling=3 (Ehrenfeucht dataset + manual N floor) in
# the earlier *uniform-coefficient forward-check* sweep -- but the full floating/grounded-BC
# inversion sensit sweep told a different story: coupling=2 has a specific, severe pathology at
# certain coefficient cells (cf101=cf103=10 and cf101=cf103=1000 both spiked to vel_rmse~13,100,
# ~13x every other cell) that the simpler forward-check never happened to probe. coupling=3 was
# clean and outlier-free across the entire 25-cell grid (vel_rmse 962-1385, no spikes). Reverted
# to coupling=3 on the strength of that full-grid evidence. AIS3_param.nc itself is also built
# with friction_coupling=3 (see ais_0.1_param.py), so this now matches the param-file default.
friction_coupling = 3

# m1qn3's relative gradient-norm stopping tolerance (default 1e-4). ROOT CAUSE of every earlier
# p=q=1 "convergence" that silently never fit anything: under p=1's much gentler cost-function
# landscape than p=3's, ||g(X)||/||g(X0)|| falls below 1e-4 by iteration ~16, while the cost is
# still falling fast (not flattening) and the fit is nowhere near done -- a false stop, not a
# real one. Tightened well below anything that can trigger this early, so every successful run
# in this pipeline instead stops on dxmin (step-size), the genuine convergence criterion.
friction_inv_gttol = 1e-8

# Dirichlet-pin two known-unstable regions (Institute/Moller Ice Stream band + an isolated
# cluster near x~350km,y~-1933km) to observed velocity during the friction inversion, instead
# of leaving them free -- mirrors Felicity's own constrain_Budd.exp/constrain_Schoof.exp
# pattern (runme.m: Inversion_Friction_Budd/Schoof). Built from diag_schoof_blowup_v2.py's
# worst-25 grounded vertices (all sat at the 100m thickness floor with driving stress
# exceeding Cmax*N under Schoof -- see docs/inversion_worklog.md). Root cause of that specific
# blowup turned out to be the forced-Newton solver setting (isnewton=2), not geometry, so this
# flag is OFF by default until an actual A/B test shows it changes anything for Budd -- see
# ssa_friction_inv_reg_lcurve below.
use_constrain_regions = False
constrain_exp_file = '/g/data/au88/jh7060/ACCESS-AIS3/assets/constrain_Budd.exp'

# Grounded-ice friction C field, in two stages (see ssa_friction_inv_lcurve /
# ssa_friction_inv_reg_lcurve below):
#   1. `friction_baseline_run` -- the UNREGULARISED (cf501 effectively off) p=q=1 baseline,
#      grounded RMSE 61.4, used only as the warm-start for stage 2 below (its own C field is
#      usable but ~5x rougher, C-field roughness 0.82 vs the p=q=3 baseline's 0.17).
#   2. `friction_lcurve_run` -- warm-started from (1), light DragCoefficientAbsGradient
#      regularisation (cf501). cf501=0.0001 is the validated corner: it drops the C-field
#      roughness to 0.18 (matching p=q=3) while the RMSE *improves* further, to 60.4 -- not a
#      tradeoff, both axes move the same direction. This is the field `ssa_inverted_solve` uses.
#
# The Budd naming pattern above (run_001_{cf101}_{cf103}_{cf501}) is specific to the Budd
# L-curve sweep; it does not apply to the Schoof m1qn3 continuation run (different control
# parameter, different script, not a cf501 grid point), so this is branched on friction_law
# rather than reused. See friction_law in ais_0.1_param.py for the full rationale for the
# current Schoof choice; ssa_friction_inv_reg_lcurve has never actually been run for Schoof --
# this points at the scratchpad-run tight-restol result saved into this same directory
# structure by finalize_schoof_friction_result.py, not a production-pipeline output.
if friction_law == 'schoof':
    friction_baseline_run = 'schoof_m1qn3_tightrestol_cmax2.0'
    friction_lcurve_run = 'schoof_m1qn3_tightrestol_cmax2.0'
else:
    friction_baseline_run = f'run_001_{friction_cf101}_{friction_cf103}_1e-08'
    friction_lcurve_run = f'run_001_{friction_cf101}_{friction_cf103}_0.0001'

## Initialise data catalog

In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

## Stage 2 (SCAFFOLD): Higher-order friction re-inversion

In [ ]:
# SSA-tuned friction (from ssa_friction_inv_reg_lcurve) isn't physically valid once HO
# adds vertical-shear resistance to the force balance -- the friction coefficient has to
# absorb a different share of the driving stress. Warm-start from the SSA-inverted C
# (same warm-start pattern validated for every mesh/geometry change this session) rather
# than cold-starting from C_init, and reuse the existing sensitivity-sweep infrastructure,
# which is confirmed to just call a generic Stressbalance solve (respects whatever flow
# equation is already set on md, not SSA-hardcoded) -- but this combination (HO +
# m1qn3 inversion via parameter_sensitivity) has not been tested. Verify on the same small
# test region as stage 1 before trusting continent-wide.
if 'ho_friction_inv' in steps:

    print("-------------------------------------------------------------")
    print(f" HIGHER-ORDER (HO) FRICTION RE-INVERSION"                  )
    print("-------------------------------------------------------------")

    print(f"-- Loading thermal steady-state model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_thermal_steadystate.nc')

    # Warm start is ALREADY carried through, not a separate step: AIS3_inverted.nc has the
    # correctly SSA-solved FrictionCoefficient, ho_thermal_steadystate extrudes it, and
    # Model.extrude() projects every existing 2D field (including friction, via
    # md.friction._extrude(md), confirmed in pyISSM/model/Model.py) onto the new 3D mesh
    # automatically -- so md.friction.coefficient here is already the warm-started field,
    # just replicated up every vertical column rather than defined only at the base.
    fric_control, fric_field, fric_min, fric_max = friction_law_info(md)

    print(f"-- Restricting friction control to base-layer vertices...")
    # Friction is a basal boundary condition -- only vertexonbase vertices are physically
    # meaningful control points. Pin every non-base vertex's bounds to its current
    # (replicated) value so m1qn3 doesn't spend gradient steps on physically meaningless
    # upper-layer copies of the same nominal field. vertexonbase / numberofvertices2d are
    # both set by Model.extrude() (pyISSM/model/Model.py:819,832).
    nv3d = md.mesh.numberofvertices
    on_base = np.asarray(md.mesh.vertexonbase).astype(bool).ravel()
    print(f"   {int(on_base.sum())} of {nv3d} vertices are on the base layer "
          f"(expect == numberofvertices2d = {md.mesh.numberofvertices2d})")

    _fld = getattr(md.friction, fric_field).astype(float)
    md.inversion = pyissm.model.classes.inversion.m1qn3(md.inversion)
    md.inversion.iscontrol = 1
    md.inversion.control_parameters = [fric_control]
    md.inversion.min_parameters = np.where(on_base, fric_min, _fld)
    md.inversion.max_parameters = np.where(on_base, fric_max, _fld)
    md.inversion.maxsteps = 500
    md.inversion.maxiter = 500
    md.inversion.gttol = friction_inv_gttol

    print(f"-- Assigning cluster and updating settings...")
    md.miscellaneous.name = 'AIS3_ho_friction_inv'
    # BUGFIX: same submission-never-happens pattern found and fixed in ssa_inverted_solve,
    # ssa_relaxation, and ho_thermal_steadystate -- waitonlock=0 + load_only=True in the
    # "if save" branch never submits (pyissm/model/execute.py:1078-1082 unconditional
    # early return), it only loads results from an already-finished prior run. Fixed to
    # the same synchronous submit-and-wait pattern used everywhere else in this pipeline.
    # Also applying the hugemem fix proactively (found necessary for ho_thermal_steadystate's
    # plain HO stress balance solve at this same 23.8M-node scale, which OOM-killed at
    # 190GB/normal-queue) -- an m1qn3 inversion adds adjoint/gradient computation on top of
    # the forward solve, so it will need at least as much memory, likely more.
    md.cluster = pyissm.model.classes.cluster.gadi()
    md.cluster.codepath = cluster.codepath
    md.cluster.executionpath = cluster.executionpath
    md.cluster.storage = cluster.storage
    md.cluster.moduleuse = cluster.moduleuse
    md.cluster.moduleload = cluster.moduleload
    md.cluster.login = cluster.login
    md.cluster.project = cluster.project
    md.cluster.queue = 'hugemem'
    md.cluster.np = 48
    md.cluster.memory = 1450  # confirmed accepted (1470-1500GB range) for ho_thermal_steadystate
    md.cluster.time = 60 * 48
    md.settings.waitonlock = 1440  # minutes

    print(f"-- Setting-up cost function coefficients/masks...")
    # Start from the validated SSA values (cf101=10/cf103=100, grounded+observed mask,
    # grounding-line-excluded 501 mask) -- same physical misfit/regularisation this project
    # has used throughout, just re-evaluated under HO's own velocity field. cf501 is kept
    # at the SSA-chosen 0.0001 as a starting point; HO's different force balance could
    # shift the misfit-vs-regularisation tradeoff, so treat this as a prior, not a final
    # value -- worth a small re-sweep (mirroring ssa_friction_inv_reg_lcurve's methodology)
    # once this converges once and a baseline RMSE is in hand.
    # BUGFIX (found via the PIG/Thwaites small-region test): cost functions 101/103 are
    # 'SurfaceAbsVelMisfit'/'SurfaceLogVelMisfit' (pyissm/model/inversions.py:13-14) --
    # velocity misfit against satellite-observed SURFACE speed, evaluated at surface
    # vertices in a 3D model, not basal ones. This scaffold reused on_base (correct for
    # the friction control's own bounds, since friction is a basal parameter) for the
    # cost-function weighting mask too, which put nonzero coefficients at vertices where
    # the misfit is never evaluated -- confirmed by the test showing contributions of
    # exactly 0 for both 101 and 103 in the printed cost table (m1qn3 was optimizing pure
    # regularisation smoothness, never actually fitting observed velocity at all). The 501
    # regularisation term (DragCoefficientAbsGradient) genuinely is a basal-quantity
    # gradient, so its mask correctly stays on_base.
    on_surface = np.asarray(md.mesh.vertexonsurface).astype(bool).ravel()
    vo = np.asarray(md.inversion.vel_obs).ravel()
    ol = np.asarray(md.mask.ocean_levelset).ravel()
    il = np.asarray(md.mask.ice_levelset).ravel()
    mask = (vo > 0) & (ol >= 0) & on_surface
    grounded_mask = (ol >= 0) & on_base
    _elx = np.asarray(md.mesh.elements).astype(int) - 1
    _float_v = ol < 0
    _touch_f = _float_v[_elx].any(axis = 1)
    _gladj = np.zeros(nv3d, dtype = bool)
    _gladj[_elx[_touch_f].ravel()] = True
    reg_mask = grounded_mask & ~_gladj

    cf = np.zeros((nv3d, 3))
    cf[mask, 0] = friction_cf101
    cf[mask, 1] = friction_cf103
    cf[reg_mask, 2] = 0.0001
    md.inversion.cost_functions = [101, 103, 501]
    md.inversion.cost_functions_coefficients = cf

    md.transient = pyissm.model.classes.transient.deactivate_all(md.transient)
    md.stressbalance.restol = 0.01
    md.stressbalance.reltol = 0.1
    md.stressbalance.abstol = np.nan
    md.settings.solver_residue_threshold = 1e-3

    if save:
        print(f"-- Submitting and waiting on HO friction inversion...")
        md = pyissm.model.execute.solve(md, 'Stressbalance', load_only = False, runtime_name = False)

        if diagnostics:
            vel = md.results.StressbalanceSolution.Vel
            gr = (il < 0) & (ol > 0) & on_base
            print(f"\nHO FRICTION INVERSION DIAGNOSTICS:")
            print(f"   Grounded (base-layer) RMSE: {np.sqrt(np.nanmean((vel[gr]-vo[gr])**2)):.2f} m/yr")

        print(f"\nSaving to {model_dir}/AIS3_ho_friction_inv.nc")
        pyissm.model.io.save_model(md, f'{model_dir}/AIS3_ho_friction_inv.nc')
    else:
        print(f"-- Submitting HO friction inversion...")
        md = pyissm.model.execute.solve(md, 'Stressbalance', load_only = False, runtime_name = False)